# Phase 2 — Data Pipeline (Two-Stage: SFT + ORPO)
### Llama-3.2-3B SFT & Alignment Project

**Goal**: Build a Quality-First, Two-Stage dataset preparation pipeline:
1. **Track A (SFT Data)**: Formats core skills, instruction following, and multi-step reasoning into unified `Alpaca-style` prompts (`text`).
2. **Track B (ORPO Data)**: Formats preference pairs (`prompt`, `chosen`, `rejected`) for alignment, safety, and hallucination reduction.
3. **Eval Safeguard**: Explicitly excludes benchmark datasets (**MMLU**, **CommonsenseQA**, **TruthfulQA**) to prevent contamination.

---

## 🗺️ Pipeline Architecture & Dataset Selection

### Track A: SFT Datasets (Skills & Reasoning)
| Dataset | HF Path | Focus |
|---|---|---|
| **OpenHermes-2.5** | `teknium/OpenHermes-2.5` | Replaces Alpaca with GPT-4 tier instruction & reasoning data |
| **Dolly-15k** | `databricks/databricks-dolly-15k` | High-trust, human-written instruction responses |
| **GSM8K** | `openai/gsm8k` | Grade-school math reasoning |
| **MetaMathQA** | `meta-math/MetaMathQA` | Multi-step mathematical reasoning |
| **HotpotQA** | `hotpot_qa` | Multi-hop contextual QA |

### Track B: ORPO Datasets (Preference & Alignment)
| Dataset | HF Path | Preference Pair Focus |
|---|---|---|
| **UltraFeedback Binarized** | `HuggingFaceH4/ultrafeedback_binarized` | General response quality & helpfulness |
| **HH-RLHF (Harmless)** | `Anthropic/hh-rlhf` | Safety alignment & harmless refusal preference |
| **HaluEval** | `pminervini/HaluEval` | Truthfulness vs. hallucinated responses |

### 🚫 Benchmark Contamination Safeguard (Eval-Only)
Strictly set aside for evaluation (not included in training):
- `cais/mmlu`
- `commonsense_qa`
- `domenicrosati/TruthfulQA` / `truthful_qa`

> No GPU needed — all cells in this notebook run on CPU.

## Setup

In [ ]:
!pip install transformers datasets huggingface_hub -q

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## CONFIG — Switch between small and full run here

Change `SMALL_RUN = True` to `False` when ready for full dataset generation.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — Edit this cell to switch between small and full runs
# ─────────────────────────────────────────────────────────────────────────────

SMALL_RUN        = True               # True = validation run, False = full production run
MAX_SAMPLES_SFT  = 3_000 if SMALL_RUN else 10_000  # max per SFT dataset
MAX_SAMPLES_ORPO = 2_000 if SMALL_RUN else 7_500   # max per ORPO dataset
MAX_SEQ_LEN      = 512                # max token cutoff

HF_USERNAME      = "your-username"    # <-- Change to your HuggingFace username
REPO_SFT         = f"{HF_USERNAME}/llama3.2-3b-sft-dataset"
REPO_ORPO        = f"{HF_USERNAME}/llama3.2-3b-orpo-dataset"

print(f"Mode            : {'SMALL (Validation)' if SMALL_RUN else 'FULL (Production)'}")
print(f"SFT Max/dataset : {MAX_SAMPLES_SFT:,}")
print(f"ORPO Max/dataset: {MAX_SAMPLES_ORPO:,}")
print(f"Target HF Repos : \n  SFT  -> {REPO_SFT}\n  ORPO -> {REPO_ORPO}")

## Load Tokenizer

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset, load_dataset
import random
import re

MODEL_ID  = "meta-llama/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
EOS       = tokenizer.eos_token   # '<|end_of_text|>'

print(f"EOS token: '{EOS}' (ID: {tokenizer.eos_token_id})")

## Shared Formatters

### SFT Prompt Formatter (`text` column)
Formats instructions into a clean Alpaca-style text block.

### ORPO Pair Formatter (`prompt`, `chosen`, `rejected` columns)
Formats preference pairs specifically for `ORPOTrainer`.

In [ ]:
def make_sft_prompt(instruction: str, response: str, input_text: str = None) -> str:
    """Unified SFT Alpaca-style training prompt."""
    instruction = str(instruction).strip()
    response    = str(response).strip()
    if input_text and str(input_text).strip():
        return (
            f"Instruction:\n{instruction}\n\n"
            f"### Input:\n{str(input_text).strip()}\n\n"
            f"### Response:\n{response}{EOS}"
        )
    return f"Instruction:\n{instruction}\n\n### Response:\n{response}{EOS}"


def make_orpo_pair(instruction: str, chosen: str, rejected: str, input_text: str = None) -> dict:
    """
    Formats a preference pair for ORPOTrainer.
    ORPOTrainer expects dict with 'prompt', 'chosen', and 'rejected'.
    """
    instruction = str(instruction).strip()
    if input_text and str(input_text).strip():
        prompt_str = f"Instruction:\n{instruction}\n\n### Input:\n{str(input_text).strip()}\n\n### Response:\n"
    else:
        prompt_str = f"Instruction:\n{instruction}\n\n### Response:\n"
    
    return {
        "prompt": prompt_str,
        "chosen": f"{str(chosen).strip()}{EOS}",
        "rejected": f"{str(rejected).strip()}{EOS}",
    }


def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

def preview_raw(dataset, n=2):
    print(f"Columns: {dataset.column_names} | Total: {len(dataset):,}")
    for i, s in enumerate(dataset.select(range(min(n, len(dataset))))):
        print(f"--- Sample {i+1} ---")
        for k, v in s.items():
            val_str = str(v)
            print(f"  {k}: {val_str[:150]}{'...' if len(val_str) > 150 else ''}")
        print()

---
#  TRACK A — SFT Dataset Curation (Skills & Reasoning)

We accumulate formatted SFT samples into `sft_dataset_list`.

In [ ]:
sft_dataset_list = []

### SFT 1 — OpenHermes-2.5 (`teknium/OpenHermes-2.5`)
Replaces Alpaca with high-quality GPT-4 tier instruction and reasoning data.

In [ ]:
print("Loading teknium/OpenHermes-2.5...")
raw_hermes = load_dataset("teknium/OpenHermes-2.5", split="train")
preview_raw(raw_hermes)

def format_hermes(sample):
    # OpenHermes has 'conversations' field or 'instruction'/'output'
    if "conversations" in sample and sample["conversations"]:
        convs = sample["conversations"]
        user_msg = next((c["value"] for c in convs if c.get("from") in ["human", "user"]), None)
        gpt_msg  = next((c["value"] for c in convs if c.get("from") in ["gpt", "assistant"]), None)
        if user_msg and gpt_msg:
            return make_sft_prompt(user_msg, gpt_msg)
    elif "instruction" in sample and "output" in sample:
        return make_sft_prompt(sample["instruction"], sample["output"], sample.get("input"))
    return None

hermes_subset = raw_hermes.shuffle(seed=42).select(range(min(MAX_SAMPLES_SFT * 2, len(raw_hermes))))
hermes_formatted = []
for s in hermes_subset:
    res = format_hermes(s)
    if res:
        hermes_formatted.append({"text": res})
    if len(hermes_formatted) >= MAX_SAMPLES_SFT:
        break

print(f"Formatted OpenHermes-2.5 samples: {len(hermes_formatted):,}")
sft_dataset_list.append(hermes_formatted)

### SFT 2 — Dolly-15k (`databricks/databricks-dolly-15k`)
Human-written instruction following across diverse categories.

In [ ]:
print("Loading databricks/databricks-dolly-15k...")
raw_dolly = load_dataset("databricks/databricks-dolly-15k", split="train")

def format_dolly(sample):
    return make_sft_prompt(sample["instruction"], sample["response"], sample.get("context"))

dolly_subset = raw_dolly.shuffle(seed=42).select(range(min(MAX_SAMPLES_SFT, len(raw_dolly))))
dolly_formatted = [{"text": format_dolly(s)} for s in dolly_subset]
print(f"Formatted Dolly-15k samples: {len(dolly_formatted):,}")
sft_dataset_list.append(dolly_formatted)

### SFT 3 — GSM8K (`openai/gsm8k`)
Grade-school math word problems with step-by-step reasoning.

In [ ]:
print("Loading openai/gsm8k...")
raw_gsm8k = load_dataset("openai/gsm8k", "main", split="train")

def format_gsm8k(sample):
    instruction = f"Solve the following math problem step by step.\n{sample['question']}"
    return make_sft_prompt(instruction, sample["answer"])

gsm8k_subset = raw_gsm8k.shuffle(seed=42).select(range(min(MAX_SAMPLES_SFT, len(raw_gsm8k))))
gsm8k_formatted = [{"text": format_gsm8k(s)} for s in gsm8k_subset]
print(f"Formatted GSM8K samples: {len(gsm8k_formatted):,}")
sft_dataset_list.append(gsm8k_formatted)

### SFT 4 — MetaMathQA (`meta-math/MetaMathQA`)
Augmented multi-step mathematical reasoning.

In [ ]:
print("Loading meta-math/MetaMathQA...")
raw_metamath = load_dataset("meta-math/MetaMathQA", split="train")

def format_metamath(sample):
    return make_sft_prompt(sample["query"], sample["response"])

metamath_subset = raw_metamath.shuffle(seed=42).select(range(min(MAX_SAMPLES_SFT, len(raw_metamath))))
metamath_formatted = [{"text": format_metamath(s)} for s in metamath_subset]
print(f"Formatted MetaMathQA samples: {len(metamath_formatted):,}")
sft_dataset_list.append(metamath_formatted)

### SFT 5 — HotpotQA (`hotpot_qa`)
Multi-hop reasoning across multiple passages.

In [ ]:
print("Loading hotpot_qa...")
raw_hotpot = load_dataset("hotpot_qa", "distractor", split="train")

def format_hotpotqa(sample):
    titles    = sample["context"]["title"]
    sentences = sample["context"]["sentences"]
    passages  = [f"{t}: " + " ".join(s) for t, s in zip(titles, sentences)]
    context_text = "\n".join(passages)
    return make_sft_prompt(sample["question"], sample["answer"], context_text)

hotpot_subset = raw_hotpot.shuffle(seed=42).select(range(min(MAX_SAMPLES_SFT, len(raw_hotpot))))
hotpot_formatted = [{"text": format_hotpotqa(s)} for s in hotpot_subset]
print(f"Formatted HotpotQA samples: {len(hotpot_formatted):,}")
sft_dataset_list.append(hotpot_formatted)

---
# TRACK B — ORPO Preference Dataset Curation (Alignment & Preference)

We accumulate preference pairs into `orpo_dataset_list`.

In [ ]:
orpo_dataset_list = []

### ORPO 1 — UltraFeedback Binarized (`HuggingFaceH4/ultrafeedback_binarized`)
General response quality and helpfulness preferences.

In [ ]:
print("Loading HuggingFaceH4/ultrafeedback_binarized...")
raw_uf = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
preview_raw(raw_uf)

def format_ultrafeedback(sample):
    prompt_text = sample.get("prompt", "")
    chosen_msgs = sample.get("chosen", [])
    rejected_msgs = sample.get("rejected", [])
    
    chosen_text = chosen_msgs[-1]["content"] if isinstance(chosen_msgs, list) and len(chosen_msgs) > 0 else str(chosen_msgs)
    rejected_text = rejected_msgs[-1]["content"] if isinstance(rejected_msgs, list) and len(rejected_msgs) > 0 else str(rejected_msgs)
    
    if not prompt_text or not chosen_text or not rejected_text:
        return None
    return make_orpo_pair(prompt_text, chosen_text, rejected_text)

uf_subset = raw_uf.shuffle(seed=42).select(range(min(MAX_SAMPLES_ORPO * 2, len(raw_uf))))
uf_formatted = []
for s in uf_subset:
    res = format_ultrafeedback(s)
    if res:
        uf_formatted.append(res)
    if len(uf_formatted) >= MAX_SAMPLES_ORPO:
        break

print(f"Formatted UltraFeedback pairs: {len(uf_formatted):,}")
orpo_dataset_list.append(uf_formatted)

### ORPO 2 — HH-RLHF Harmless (`Anthropic/hh-rlhf`)
Safety alignment & harmless refusal preference pairs.

In [ ]:
print("Loading Anthropic/hh-rlhf (harmless subset)...")
raw_hhrlhf = load_dataset("Anthropic/hh-rlhf", split="train")

def parse_hhrlhf_pair(text: str):
    parts = re.split(r'\n\nHuman: |\n\nAssistant: ', text)
    parts = [p.strip() for p in parts if p.strip()]
    if len(parts) < 2:
        return None, None
    return parts[0], parts[-1]

def format_hhrlhf_orpo(sample):
    prompt_c, chosen_resp = parse_hhrlhf_pair(sample["chosen"])
    prompt_r, rejected_resp = parse_hhrlhf_pair(sample["rejected"])
    prompt = prompt_c or prompt_r
    if not prompt or not chosen_resp or not rejected_resp:
        return None
    return make_orpo_pair(prompt, chosen_resp, rejected_resp)

hh_subset = raw_hhrlhf.shuffle(seed=42).select(range(min(MAX_SAMPLES_ORPO * 2, len(raw_hhrlhf))))
hh_formatted = []
for s in hh_subset:
    res = format_hhrlhf_orpo(s)
    if res:
        hh_formatted.append(res)
    if len(hh_formatted) >= MAX_SAMPLES_ORPO:
        break

print(f"Formatted HH-RLHF pairs: {len(hh_formatted):,}")
orpo_dataset_list.append(hh_formatted)

### ORPO 3 — HaluEval (`pminervini/HaluEval`)
Truthfulness vs. hallucinated responses.

In [ ]:
print("Loading pminervini/HaluEval (qa_data)...")
try:
    raw_halu = load_dataset("pminervini/HaluEval", "qa_data", split="train")
except Exception:
    raw_halu = load_dataset("pminervini/HaluEval", split="train")

def format_halueval(sample):
    question = sample.get("question") or sample.get("user_query", "")
    chosen   = sample.get("right_answer") or sample.get("ground_truth", "")
    rejected = sample.get("hallucinated_answer") or sample.get("hallucinated_response", "")
    ctx      = sample.get("knowledge") or sample.get("domain", None)
    if not question or not chosen or not rejected:
        return None
    return make_orpo_pair(question, chosen, rejected, input_text=ctx)

halu_subset = raw_halu.shuffle(seed=42).select(range(min(MAX_SAMPLES_ORPO * 2, len(raw_halu))))
halu_formatted = []
for s in halu_subset:
    res = format_halueval(s)
    if res:
        halu_formatted.append(res)
    if len(halu_formatted) >= MAX_SAMPLES_ORPO:
        break

print(f"Formatted HaluEval pairs: {len(halu_formatted):,}")
orpo_dataset_list.append(halu_formatted)

---
# Benchmark Contamination Safeguard Notice

The following evaluation benchmarks are **strictly excluded** from both Track A and Track B to maintain evaluation integrity:
- `cais/mmlu` (Knowledge Breadth)
- `commonsense_qa` (Commonsense Reasoning)
- `domenicrosati/TruthfulQA` (Truthfulness & Misconceptions)

---
# Combine, Filter & Analyze Datasets

In [ ]:
# Flatten SFT & ORPO dataset lists
sft_combined  = [item for ds in sft_dataset_list for item in ds]
orpo_combined = [item for ds in orpo_dataset_list for item in ds]

random.seed(42)
random.shuffle(sft_combined)
random.shuffle(orpo_combined)

print(f"Total raw SFT samples : {len(sft_combined):,}")
print(f"Total raw ORPO pairs  : {len(orpo_combined):,}")

# Token length filtering
sft_filtered  = [s for s in sft_combined if count_tokens(s["text"]) <= MAX_SEQ_LEN]
orpo_filtered = [
    p for p in orpo_combined 
    if count_tokens(p["prompt"] + p["chosen"]) <= MAX_SEQ_LEN and count_tokens(p["prompt"] + p["rejected"]) <= MAX_SEQ_LEN
]

print(f"\nFiltered SFT samples  (<= {MAX_SEQ_LEN} tokens): {len(sft_filtered):,}")
print(f"Filtered ORPO pairs    (<= {MAX_SEQ_LEN} tokens): {len(orpo_filtered):,}")

---
# Push Datasets to HuggingFace Hub

In [ ]:
sft_hf  = Dataset.from_list(sft_filtered).train_test_split(test_size=0.02, seed=42)
orpo_hf = Dataset.from_list(orpo_filtered).train_test_split(test_size=0.02, seed=42)

print(f"SFT dataset  -> Train: {len(sft_hf['train']):,}, Test: {len(sft_hf['test']):,}")
print(f"ORPO dataset -> Train: {len(orpo_hf['train']):,}, Test: {len(orpo_hf['test']):,}")

# Push SFT dataset
sft_hf.push_to_hub(REPO_SFT, private=True)
print(f"\n✅ SFT Dataset pushed to: https://huggingface.co/datasets/{REPO_SFT}")

# Push ORPO dataset
orpo_hf.push_to_hub(REPO_ORPO, private=True)
print(f"✅ ORPO Dataset pushed to: https://huggingface.co/datasets/{REPO_ORPO}")

## Phase 2 Complete

| Output | HF Repository | Purpose |
|---|---|---|
| **Track A (SFT)** | `{REPO_SFT}` | Stage 1 QLoRA SFT training |
| **Track B (ORPO)** | `{REPO_ORPO}` | Stage 2 QLoRA ORPO preference alignment |

### Next Steps
- **Phase 3**: Set up Unsloth QLoRA config.
- **Phase 4**: Stage 1 QLoRA SFT training run.
- **Phase 4.5**: Stage 2 QLoRA ORPO preference alignment run.